# LeetCode #93: Restore IP Addresses

https://leetcode.com/problems/restore-ip-addresses/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(3^4) = O(1)$ | $O(1)$ |
| **Optimal: Backtracking ★** | $O(3^4) = O(1)$ | $O(1)$ |

---

## Understanding the Methods

### Brute Force
Use three nested loops to try all possible split positions for the four segments. For each candidate, validate all segments and collect valid IPs. More verbose and harder to generalize.

### Optimal: Backtracking ★
Recursively build segments 0–3, trying lengths 1–3 at each step. Immediately prune if a segment value exceeds 255, has a leading zero (unless the segment is "0"), or if the remaining string cannot be split into the required remaining segments (too short or too long).

**Why this is better than Brute Force:** The pruning conditions cut invalid branches instantly without enumerating all combinations first; the resulting recursion visits at most $3^4 = 81$ states and is simpler to extend than triply-nested loops.

**Constraints:**
* 4 <= s.length <= 12
* s consists of digits only

## Solutions

### C#

In [ ]:
public class Solution {
    public IList<string> RestoreIpAddresses(string s) {
        var result = new List<string>();
        Backtrack(s, 0, new List<string>(), result);
        return result;
    }

    private void Backtrack(string s, int start, List<string> parts, IList<string> result) {
        if (parts.Count == 4 && start == s.Length) {
            // All four segments placed and the whole string consumed — valid IP
            result.Add(string.Join(".", parts));
            return;
        }
        if (parts.Count == 4 || start == s.Length) return;

        for (int len = 1; len <= 3; len++) {
            if (start + len > s.Length) break;
            string seg = s.Substring(start, len);
            // Prune leading zeros (e.g. "01" is invalid) and values above 255
            if (seg.Length > 1 && seg[0] == '0') break;
            if (int.Parse(seg) > 255) break;
            // Prune if remaining characters cannot fill the remaining segments
            int remaining = s.Length - (start + len);
            int segLeft = 3 - parts.Count;
            if (remaining < segLeft || remaining > segLeft * 3) continue;

            parts.Add(seg);
            Backtrack(s, start + len, parts, result);
            // Undo the segment to try the next length
            parts.RemoveAt(parts.Count - 1);
        }
    }
}

### Python

In [ ]:
class Solution:
    def restoreIpAddresses(self, s: str) -> list[str]:
        result = []

        def backtrack(start: int, parts: list[str]) -> None:
            if len(parts) == 4 and start == len(s):
                # All four segments placed and the whole string consumed — valid IP
                result.append('.'.join(parts))
                return
            if len(parts) == 4 or start == len(s):
                return

            for length in range(1, 4):
                if start + length > len(s):
                    break
                seg = s[start:start + length]
                # Prune leading zeros (e.g. "01" is invalid) and values above 255
                if len(seg) > 1 and seg[0] == '0':
                    break
                if int(seg) > 255:
                    break
                # Prune if remaining characters cannot fill the remaining segments
                remaining = len(s) - (start + length)
                seg_left = 3 - len(parts)
                if remaining < seg_left or remaining > seg_left * 3:
                    continue

                parts.append(seg)
                backtrack(start + length, parts)
                # Undo the segment to try the next length
                parts.pop()

        backtrack(0, [])
        return result

### Go

In [ ]:
func restoreIpAddresses(s string) []string {
    result := []string{}
    parts := []string{}

    var backtrack func(start int)
    backtrack = func(start int) {
        if len(parts) == 4 && start == len(s) {
            // All four segments placed and the whole string consumed — valid IP
            result = append(result, strings.Join(parts, "."))
            return
        }
        if len(parts) == 4 || start == len(s) {
            return
        }
        for length := 1; length <= 3; length++ {
            if start+length > len(s) {
                break
            }
            seg := s[start : start+length]
            // Prune leading zeros (e.g. "01" is invalid) and values above 255
            if len(seg) > 1 && seg[0] == '0' {
                break
            }
            val, _ := strconv.Atoi(seg)
            if val > 255 {
                break
            }
            // Prune if remaining characters cannot fill the remaining segments
            remaining := len(s) - (start + length)
            segLeft := 3 - len(parts)
            if remaining < segLeft || remaining > segLeft*3 {
                continue
            }
            parts = append(parts, seg)
            backtrack(start + length)
            // Undo the segment to try the next length
            parts = parts[:len(parts)-1]
        }
    }
    backtrack(0)
    return result
}

### Rust

In [ ]:
impl Solution {
    pub fn restore_ip_addresses(s: String) -> Vec<String> {
        let s: Vec<char> = s.chars().collect();
        let mut result = Vec::new();
        let mut parts: Vec<String> = Vec::new();
        Self::backtrack(&s, 0, &mut parts, &mut result);
        result
    }

    fn backtrack(s: &[char], start: usize, parts: &mut Vec<String>, result: &mut Vec<String>) {
        if parts.len() == 4 && start == s.len() {
            // All four segments placed and the whole string consumed — valid IP
            result.push(parts.join("."));
            return;
        }
        if parts.len() == 4 || start == s.len() { return; }

        for length in 1..=3 {
            if start + length > s.len() { break; }
            let seg: String = s[start..start+length].iter().collect();
            // Prune leading zeros (e.g. "01" is invalid) and values above 255
            if length > 1 && s[start] == '0' { break; }
            let val: u32 = seg.parse().unwrap();
            if val > 255 { break; }
            // Prune if remaining characters cannot fill the remaining segments
            let remaining = s.len() - (start + length);
            let seg_left = 3 - parts.len();
            if remaining < seg_left || remaining > seg_left * 3 { continue; }

            parts.push(seg);
            Self::backtrack(s, start + length, parts, result);
            // Undo the segment to try the next length
            parts.pop();
        }
    }
}

## Example Scenarios

**1. Common Case**
**Input:** s = "25525511135"
At depth 0, try "2" (val 2), "25" (val 25), "255" (val 255). With seg="255", remaining="25511135" (8 chars), 3 segments left (max 9 chars, min 3 chars) — valid. Recurse; eventually produces "255.255.11.135" and "255.255.111.35". Two valid IPs.

**2. Slightly Complex**
**Input:** s = "0000"
The only valid split is "0.0.0.0". For each segment, only length=1 is valid (length=2 would give "00" with a leading zero, pruned by break). Each "0" passes the value check (0 <= 255). Result: ["0.0.0.0"].

**3. Edge Case: Time Factor**
**Input:** s = "111111111111" (twelve 1s, maximum length)
At each of the 4 segment positions, lengths 1, 2, 3 are all valid (all give values 1, 11, or 111 ≤ 255). The search visits at most $3^4 = 81$ states. Even without any pruning, this is the absolute maximum; with the length-feasibility check many branches are eliminated early.

**4. Edge Case: Space Factor**
**Input:** s = "1111111111111" — wait, max is 12 chars; try s = "999999999999".
At depth 0 with length=3, seg="999" (val 999 > 255) → break immediately. Only lengths 1 and 2 survive ("9" and "99"). The `parts` list holds at most 4 strings; stack depth is at most 4. Space is $O(1)$ regardless of input.

**5. Almost-Impossible but Plausible**
**Input:** s = "010010" (contains leading zeros in all non-trivial splits)
Valid segments must avoid leading zeros except for the lone "0". The only valid splits produce "0.10.0.10" and "0.100.1.0". Attempts at "01" are pruned immediately. The feasibility guard further cuts branches where remaining length is out of range for the remaining segments.